In [133]:
import torch

print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [134]:
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
pp = '../data/custom_music/hunting_soul.webm'

In [135]:
# for data transformation 
import numpy as np 
# for visualizing the data 
import matplotlib.pyplot as plt 
# for opening the media file 
import scipy.io.wavfile as wavfile

In [136]:
# Step 0: convert to wav if not in the format
from moviepy import VideoFileClip
clip = VideoFileClip(pp)
clip.audio.write_audiofile("out_audio.wav")


MoviePy - Writing audio in out_audio.wav


MoviePy - Done.


In [137]:
# Step 1: load wav
import librosa
import librosa.display
import matplotlib.pyplot as plt
y, sr = librosa.load('out_audio.wav') # sr = sample rate
y = y[:int(sr*30)]


In [138]:
# Step 2: do fft over the audio file and convert output to mel scale
mel_spect = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=2048, hop_length=1024)
# # COnvert amplitude to deibels so that difference in loudness is distinguishable
mel_spect = librosa.power_to_db(mel_spect, ref=np.max)
# getting all values between 0 and 1
mel_spect = (mel_spect - mel_spect.min()) / (mel_spect.max() - mel_spect.min())

In [139]:
mel_spect.shape

(128, 646)

In [177]:
# code for spectogram testing(take file input->check whether it is wav or not(webm)->if webm->convert to wav using moviepy->else generate spectrogram and return mel array)
import numpy as np
import matplotlib.pyplot as plt
from moviepy import VideoFileClip
import librosa
N_FFT = 6767
HOP_LEN = N_FFT
def generate_spectrogram(file_path):
    file_path =  "../data/genres_original/" + file_path[:file_path.find(".")] + "/" + file_path
    # some file in data set are corrupted -_-
    try:
        
        if(".wav" not in file_path):
            # Step 1: convert to wav if not in the format
            clip = VideoFileClip(file_path)
            clip.audio.write_audiofile("out_audio.wav")
            file_path="out_audio.wav"
        y, sr = librosa.load(file_path) #sr sample rate
        # some sample smaller than 30s
        if len(y) < sr*30:
            y = np.pad(y, (0, int(sr*30) - len(y)))
        y = y[:int(sr*30)]
        # Step 2: do fft over the audio file and convert output to mel scale
        mel_spect = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LEN)
        # # Convert amplitude to decibels so that difference in loudness is distinguishable
        mel_spect = librosa.power_to_db(mel_spect, ref=np.max)
        # getting all values between 0 and 1
        mel_spect = (mel_spect - mel_spect.min()) / (mel_spect.max() - mel_spect.min())
        return mel_spect
    except Exception:
        print(file_path)
        print("Audio corrupt.. skip")
        return None

In [178]:
import os
from sklearn.preprocessing import LabelEncoder
import numpy as np

def preprocess():
    data = "../data/genres_original"
    x = []
    y = []

    # loop over folder and sub-folder
    for genre_folder in os.scandir(data):
        for files in os.scandir(genre_folder):
            # get sample
            sample = files.name
            spectogram = generate_spectrogram(sample)
            # if corrupt then dont add
            if spectogram is not None:
                x.append(spectogram)
                y.append(sample[:sample.find(".")])
    
    # get string to num maping
    mapping = LabelEncoder()
    y = mapping.fit_transform(y)
    mapping = dict(zip(range(len(mapping.classes_)), mapping.classes_))
    x = np.array(x)
    
    
    return x,y,mapping

x,y,mapping = preprocess()

C:\Users\Priyam Patel\AppData\Local\Temp\ipykernel_16440\4291103695.py:18: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path) #sr sample rate
c:\Users\Priyam Patel\PRML-Project\.venv\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


../data/genres_original/jazz/jazz.00054.wav
Audio corrupt.. skip


In [179]:
# convert to torch format which expects [batchsize, channels, height, width]
x_tensor = x[:, np.newaxis,:,:]
x_tensor = torch.tensor(x_tensor, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)
x_tensor.shape

torch.Size([999, 1, 128, 98])

In [180]:
from torch.utils.data import DataLoader, TensorDataset
dataset = TensorDataset(x_tensor, y_tensor)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

In [181]:
from sklearn.model_selection import train_test_split
x_train, x_val, y_train, y_val = train_test_split(
    x_tensor, y_tensor,
    test_size=0.1,
    random_state=68097,
    stratify=y_tensor,
)

traindataset = TensorDataset(x_train, y_train)
trainloader = DataLoader(traindataset, batch_size=4, shuffle=True)

valdataset = TensorDataset(x_val, y_val)
valloader = DataLoader(valdataset, batch_size=4, shuffle=True)

In [184]:
# ==============================================================================
# BASIC CNN ARCHITECTURE
# ==============================================================================
from torch.optim import lr_scheduler

class BasicCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(BasicCNN, self).__init__()

        # FEATURE EXTRACTOR: Learns spatial patterns (edges, textures, shapes)
        self.features = nn.Sequential(
            # Layer 1: Conv -> ReLU -> Pool
            # Input: 3 channels (RGB), 32x32 image. Output: 32 channels, 32x32
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            # MaxPool reduces dimensions by half. Output: 32 channels, 16x16 image
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Layer 2: Conv -> ReLU -> Pool
            # Input: 32 channels, 16x16. Output: 64 channels, 16x16
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            # Output: 64 channels, 8x8 image
            nn.MaxPool2d(kernel_size=5, stride=5)
        )

        # CLASSIFIER: Takes the extracted features and makes a decision
        self.classifier = nn.Sequential(
            # Flatten the 64 channels of 8x8 images: 64 * 8 * 8 = 4096
            nn.Linear(6912, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5), # Drops 50% of neurons randomly to prevent memorization (overfitting)
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)         # 1. Pass through convolutional layers
        x = torch.flatten(x, 1)      # 2. Flatten from 2D grids into a 1D vector
        x = self.classifier(x)       # 3. Pass through dense/linear layers
        return x

In [190]:
model = BasicCNN()
model = model.to("cuda")
criterion = nn.CrossEntropyLoss()
optimizer  = optim.Adam(model.parameters(), 0.0002)
train_losses = []
device = "cuda"
print("Starting Training...")
epo = 80
for epoch in range(epo):
    model.train()
    running_loss = 0.0

    for x, y in trainloader:
        x, y = x.to("cuda"), y.to("cuda")

        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_train_loss = running_loss / len(trainloader)
    train_losses.append(avg_train_loss)
    

    print(f"Epoch [{epoch+1}/{epo}], Loss: {avg_train_loss:.4f},")

    # Evaluate model on test data
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in valloader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            _, predicted = torch.max(outputs.data, 1)
            total += y.size(0)
            correct += (predicted == y).sum().item()

    test_accuracy = 100 * correct / total
    print(f"Val Accuracy: {test_accuracy:.2f}%\n")

Starting Training...
Epoch [1/80], Loss: 2.2482,
Val Accuracy: 22.00%

Epoch [2/80], Loss: 2.0803,
Val Accuracy: 22.00%

Epoch [3/80], Loss: 1.9289,
Val Accuracy: 31.00%

Epoch [4/80], Loss: 1.8311,
Val Accuracy: 34.00%

Epoch [5/80], Loss: 1.7195,
Val Accuracy: 39.00%

Epoch [6/80], Loss: 1.6143,
Val Accuracy: 46.00%

Epoch [7/80], Loss: 1.5566,
Val Accuracy: 51.00%

Epoch [8/80], Loss: 1.4667,
Val Accuracy: 46.00%

Epoch [9/80], Loss: 1.4198,
Val Accuracy: 52.00%

Epoch [10/80], Loss: 1.3897,
Val Accuracy: 60.00%

Epoch [11/80], Loss: 1.3285,
Val Accuracy: 53.00%

Epoch [12/80], Loss: 1.3045,
Val Accuracy: 60.00%

Epoch [13/80], Loss: 1.2032,
Val Accuracy: 57.00%

Epoch [14/80], Loss: 1.1519,
Val Accuracy: 54.00%

Epoch [15/80], Loss: 1.1145,
Val Accuracy: 62.00%

Epoch [16/80], Loss: 1.1156,
Val Accuracy: 55.00%

Epoch [17/80], Loss: 1.0604,
Val Accuracy: 61.00%

Epoch [18/80], Loss: 1.0201,
Val Accuracy: 62.00%

Epoch [19/80], Loss: 1.0225,
Val Accuracy: 58.00%

Epoch [20/80], Loss